# 🌅 Morning Market Update

### Gmail App Password
Enable 2FA on your Google account, then go to:
https://myaccount.google.com/apppasswords
Create a password for 'Morning Dashboard' and paste the 16-char code below.

In [87]:
import warnings
import smtplib
import os
import requests
import pandas as pd
import yfinance as yf
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

warnings.filterwarnings('ignore')

In [88]:
# Vos détails personnels

FRED_API_KEY = 'dc40184816d6802a16efc0b3b4c02761' # Please use your owm key by creating an account on https://fredaccount.stlouisfed.org

EMAIL_SENDER = 'enzo.marichal@gmail.com' # your Gmail address
EMAIL_PASSWORD = 'suxb zmrw nqkv sjxe' # Gmail App Password (NOT your real password), please check: https://myaccount.google.com/apppasswords
EMAIL_TO = 'enzo.marichal@edu.escp.eu' # recipient (can be the same address)
EMAIL_SUBJECT = '🌅 Morning Market Update'

#OUTPUT_DIR = os.getcwd()
#HTML_FILE = os.path.join(OUTPUT_DIR, 'dashboard.html') # pas obligatoire
#PDF_FILE = os.path.join(OUTPUT_DIR, 'morning_market_dashboard.pdf') # pas obligatoire

In [89]:
# Les données que l'on veut (feel free to add/amend)

EQUITY_INDICES = {
    'CAC 40':'^FCHI',
    'DAX':'^GDAXI',
    'FTSE 100':'^FTSE',
    'Eurostoxx 50':'^STOXX50E',
    'Dow Jones':'^DJI',
    'NASDAQ 100':'^NDX',
    'S&P 500':'^GSPC',
    'VIX':'^VIX',
}

FX_PAIRS = {
    'EUR/USD':'EURUSD=X',
    'EUR/GBP':'EURGBP=X',
    'GBP/USD':'GBPUSD=X',
    'USD/JPY':'JPY=X',
    'USD/CHF':'CHF=X',
    'USD/CAD':'CAD=X',
}

COMMODITIES = {
    'Gold ($/oz)':'GC=F',
    'Silver ($/oz)':'SI=F',
    'Brent Crude ($/bbl)':'BZ=F',
    'WTI Crude ($/bbl)':'CL=F',
    'Natural Gas ($/MMBtu)':'NG=F',
    'Copper ($/lb)':'HG=F',
    'Wheat (¢/bu)':'ZW=F',
    'Cocoa ($/MT)':'CC=F',
}

STOCKS = {
    'Apple':'AAPL',
    'Microsoft':'MSFT',
    'NVIDIA':'NVDA',
    'Tesla':'TSLA',
    'Palantir':'PLTR',
    'Goldman Sachs':'GS',
    'JPMorgan':'JPM',
    'Chevron':'CVX',
    'LVMH':'MC.PA',
    'Shell':'SHEL',
    'Saudi Aramco':'2222.SR',
}

US_YF_YIELDS = {
    'T-Bill 3M':'^IRX',
    '5Y Note':'^FVX',
    '10Y Note':'^TNX',
    '30Y Bond':'^TYX',
}

FRED_BOND_SERIES = {
    'US 2Y':('DGS2','daily'),
    'US 5Y':('DGS5','daily'),
    'US 10Y':('DGS10','daily'),
    'UK GILT 10Y':('IRLTLT01GBM156N','monthly'),
    'France OAT 10Y':('IRLTLT01FRM156N','monthly'),
    'Italy BTP 10Y':('IRLTLT01ITM156N','monthly'),
    'Germany Bund 10Y':('IRLTLT01DEM156N','monthly'),
    'Japan JGB 10Y':('IRLTLT01JPM156N','monthly'),
}

FRED_RATE_SERIES = {
    'FED Funds Rate':'DFF',
    'ECB Deposit Rate':'ECBDFR',
    'SOFR':'SOFR',
    'SONIA':'SONIA',
}

In [90]:
# DATA FETCH

def _yf_fetch(tickers_dict, period='5d'):
    results = {k: {'price': None, 'pct': None} for k in tickers_dict}
    symbols = list(tickers_dict.values())
    rev = {v: k for k, v in tickers_dict.items()}
    if not symbols:
        return results
    try:
        raw = yf.download(symbols, period=period, auto_adjust=True, progress=False)
        close = raw['Close']
        for sym in symbols:
            name = rev[sym]
            try:
                s = (close[sym] if isinstance(close, pd.DataFrame) else close).dropna()
                if len(s) >= 2:
                    c, p = float(s.iloc[-1]), float(s.iloc[-2])
                    results[name] = {'price': round(c, 4), 'pct': round((c - p) / p * 100, 2)}
                elif len(s) == 1:
                    results[name]['price'] = round(float(s.iloc[-1]), 4)
            except Exception:
                pass
    except Exception as e:
        print(f'  [yfinance] Error: {e}')
    return results


def _fred_value(series_id):
    if not FRED_API_KEY:
        return None, None
    try:
        r = requests.get(
            'https://api.stlouisfed.org/fred/series/observations',
            params={'series_id': series_id, 'api_key': FRED_API_KEY,
                    'file_type': 'json', 'sort_order': 'desc', 'limit': 10},
            timeout=10,
        )
        r.raise_for_status()
        for obs in r.json().get('observations', []):
            if obs['value'] != '.':
                return float(obs['value']), obs['date']
    except Exception:
        pass
    return None, None


def _ecb_yield(maturity):
    key = f'B.U2.EUR.4F.G_N_A.SV_C_YM.SR_{maturity}'
    try:
        r = requests.get(
            f'https://sdw-wsrest.ecb.europa.eu/service/data/YC/{key}',
            params={'detail': 'dataonly', 'format': 'json', 'lastNObservations': '1'},
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()
        series = data['dataSets'][0]['series']
        sk = list(series.keys())[0]
        obs = series[sk]['observations']
        latest = max(obs.keys(), key=int)
        return round(float(obs[latest][0]), 3)
    except Exception:
        return None


def fetch_all():
    ts = datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')
    print(f"\n{'═'*52}")
    print(f'  Morning Market Dashboard — {ts}')
    print(f"{'═'*52}\n")

    print('Indices...')
    indices = _yf_fetch(EQUITY_INDICES)
    print('FX...')
    fx = _yf_fetch(FX_PAIRS)
    print('Commodities...')
    commodities = _yf_fetch(COMMODITIES)
    print('Stocks...')
    stocks = _yf_fetch(STOCKS)
    print('US Treasury yields (Yahoo Finance)...')
    us_yields = _yf_fetch(US_YF_YIELDS)

    bond_yields = {}
    if FRED_API_KEY:
        print('Bond yields (FRED)...')
        for name, (sid, freq) in FRED_BOND_SERIES.items():
            v, dt = _fred_value(sid)
            bond_yields[name] = {'value': v, 'date': dt, 'freq': freq}
    else:
        print('Set FRED_API_KEY for US 2Y + European bond yields.')

    print('Euro Area yields (ECB SDW)...')
    ea_yields = {}
    for mat in ['2Y', '5Y', '10Y']:
        ea_yields[f'EA/Bund {mat}'] = _ecb_yield(mat)

    cb_rates = {}
    if FRED_API_KEY:
        print('Central bank rates (FRED)...')
        for name, sid in FRED_RATE_SERIES.items():
            v, dt = _fred_value(sid)
            cb_rates[name] = {'rate': v, 'date': dt}
    else:
        print('Set FRED_API_KEY for FED, ECB, SOFR, SONIA rates.')

    print('\nOK Data fetched.\n')
    return {
        'timestamp': ts, 'indices': indices, 'fx': fx,
        'commodities': commodities, 'stocks': stocks,
        'us_yields': us_yields, 'bond_yields': bond_yields,
        'ea_yields': ea_yields, 'cb_rates': cb_rates,
    }

In [91]:
# Rendu HTML & PDF

CSS = """
<style>
@page { size: A4 landscape; margin: 1cm; }
* { box-sizing: border-box; margin: 0; padding: 0; }
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    background: #ffffff; color: #24292f; font-size: 11px;
}
.mmd { padding: 12px; }
.mmd-hd { text-align: center; padding: 6px 0 14px; }
.mmd-hd h1 { font-size: 18px; color: #0969da; font-weight: 700; margin-bottom: 2px; }
.mmd-hd p  { font-size: 10px; color: #656d76; }
.layout-table { width: 100%; border-collapse: separate; border-spacing: 10px; }
.layout-table td { vertical-align: top; width: 50%; }
.card {
    background: #f6f8fa;
    border: 1px solid #d0d7de;
    border-radius: 6px;
    overflow: hidden;
}
.ct {
    background: #eaeef2;
    padding: 6px 10px;
    font-size: 11px;
    font-weight: 700;
    color: #0969da;
    border-bottom: 1px solid #d0d7de;
}
.card table { width: 100%; border-collapse: collapse; }
.card th {
    padding: 4px 8px;
    font-size: 9px;
    color: #656d76;
    text-align: left;
    background: #f6f8fa;
    text-transform: uppercase;
    letter-spacing: .3px;
    border-bottom: 1px solid #d0d7de;
}
.card td {
    padding: 4px 8px;
    font-size: 10px;
    border-bottom: 1px solid #eaeef2;
    white-space: nowrap;
}
.card tr:last-child td { border-bottom: none; }
.up   { color: #1a7f37; font-weight: 600; }
.down { color: #cf222e; font-weight: 600; }
.neu  { color: #656d76; }
.note { font-size: 8px; color: #656d76; font-style: italic; padding: 3px 8px 6px; display: block; }
.footer { text-align: center; font-size: 8px; color: #656d76; padding: 8px 0 2px; }
</style>
"""


def _fp(v, dec=2):
    if v is None:
        return '<span class="neu">—</span>'
    return f'{v:,.{dec}f}'


def _fpct(v):
    if v is None:
        return '<span class="neu">—</span>'
    cls = 'up' if v > 0 else ('down' if v < 0 else 'neu')
    arrow = '▲'  if v > 0 else ('▼'    if v < 0 else '')
    return f'<span class="{cls}">{arrow}{abs(v):.2f}%</span>'


def _card(title, headers, rows, note=''):
    ths = ''.join(f'<th>{h}</th>' for h in headers)
    trs = ''.join('<tr>' + ''.join(f'<td>{c}</td>' for c in row) + '</tr>' for row in rows)
    note_html = f'<span class="note">{note}</span>' if note else ''
    return (
        '<div class="card">'
        f'<div class="ct">{title}</div>'
        f'<table><thead><tr>{ths}</tr></thead><tbody>{trs}</tbody></table>'
        f'{note_html}</div>'
    )

# Titres
def build_html(data):
    c_idx = _card('Equity Indices', ['Index', 'Level', 'Chg %'],
                  [[n, _fp(d['price'], 2), _fpct(d['pct'])] for n, d in data['indices'].items()])

    c_fx  = _card('FX Rates', ['Pair', 'Rate', 'Chg %'],
                  [[n, _fp(d['price'], 4), _fpct(d['pct'])] for n, d in data['fx'].items()])

    c_com = _card('Commodities', ['Type', 'Price', 'Chg %'],
                  [[n, _fp(d['price'], 2), _fpct(d['pct'])] for n, d in data['commodities'].items()])

    c_stk = _card('Stocks', ['Company', 'Price ($)', 'Chg %'],
                  [[n, _fp(d['price'], 2), _fpct(d['pct'])] for n, d in data['stocks'].items()])

    # Bond yields
    yrows = []
    for n, d in data['us_yields'].items():
        v = (_fp(d['price'], 3) + '%') if d['price'] else '<span class="neu">—</span>'
        yrows.append([f'US {n}', v, '—', 'Yahoo Finance (daily)'])
    for n, d in data['bond_yields'].items():
        star  = ' ★' if d['freq'] == 'monthly' else ''
        v_str = (_fp(d['value'], 3) + '%') if d['value'] is not None else '<span class="neu">—</span>'
        yrows.append([n + star, v_str, d.get('date') or '—', f"FRED ({d['freq']})"])
    for n, v in data['ea_yields'].items():
        v_str = (_fp(v, 3) + '%') if v is not None else '<span class="neu">—</span>'
        yrows.append([n, v_str, '—', 'ECB SDW (daily)'])
    c_yields = _card(
        'Government Bond Yields',
        ['Instrument', 'Yield', 'As of', 'Source'],
        yrows,
        note='Monthly FRED data — refreshed once per month. EA/Bund = Euro Area AAA sovereign yield (ECB, daily) — close proxy for German Bund.'
    )

    # CB Rates
    if data['cb_rates']:
        cb_rows = [[n,
                    (_fp(d['rate'], 2) + '%' if d['rate'] is not None else '<span class="neu">—</span>'),
                    d.get('date') or '—']
                   for n, d in data['cb_rates'].items()]
    else:
        cb_rows = [['Set FRED_API_KEY to enable', '—', '—']]
    c_cb = _card('Central Bank Rates', ['Rate', 'Value', 'Date'], cb_rows)

    # Layout: HTML table (2 columns) — weasyprint-compatible
    layout = (
        '<table class="layout-table">'
        f'<tr><td>{c_idx}</td><td>{c_fx}</td></tr>'
        f'<tr><td>{c_com}</td><td>{c_stk}</td></tr>'
        f'<tr><td colspan="2">{c_yields}</td></tr>'
        f'<tr><td colspan="2">{c_cb}</td></tr>'
        '</table>'
    )

    return (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>" + CSS + "</head><body>"
        + '<div class="mmd">'
        + '<div class="mmd-hd"><h1>Morning Market Update</h1>'
        + f'<p>Data as of {data["timestamp"]}</p></div>'
        + layout
        + '<p class="footer">Yahoo Finance &nbsp;·&nbsp; ECB Statistical Data Warehouse &nbsp;·&nbsp; FRED / St. Louis Fed</p>'
        + '</div></body></html>'
    )


def html_to_pdf(html_content, pdf_path):
    try:
        from weasyprint import HTML as WH
        WH(string=html_content).write_pdf(pdf_path)
        print(f'✅ PDF generated: {pdf_path}')
        return True
    except Exception as e:
        print(f'❌ PDF generation failed: {e}')
        return False


In [92]:
# EMAIL

def send_email(pdf_path, timestamp):
    if not os.path.exists(pdf_path):
        print('❌ No PDF found — email not sent.')
        return

    msg = MIMEMultipart()
    msg['From'] = EMAIL_SENDER
    msg['To'] = EMAIL_TO
    msg['Subject'] = f'{EMAIL_SUBJECT} — {timestamp[:10]}'

    body = (
        f'Bonjour,\n\n'
        f'Veuillez trouver ci-joint le Morning Market Update du {timestamp[:10]}.\n\n'
        f'Bonne journée,\n'
        f'Enzo Marichal'
    )
    msg.attach(MIMEText(body, 'plain'))

    with open(pdf_path, 'rb') as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition',
                    f'attachment; filename=market_dashboard_{timestamp[:10]}.pdf')
    msg.attach(part)

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.sendmail(EMAIL_SENDER, EMAIL_TO, msg.as_string())
        print(f'✅ Email sent to {EMAIL_TO}')
    except Exception as e:
        print(f'❌ Email failed: {e}')
        print('Check EMAIL_SENDER, EMAIL_PASSWORD (App Password) and EMAIL_TO.')

In [93]:
# RUN : fetch, build PDF, send email

data = fetch_all()
html = build_html(data)

# Save HTML (pas obligatoire)
#with open(HTML_FILE, 'w', encoding='utf-8') as f:
    #f.write(html)
#print(f'HTML saved: {HTML_FILE}')

# Convert to PDF
import tempfile
with tempfile.NamedTemporaryFile(suffix='.pdf', delete=True) as tmp:
    PDF_FILE = tmp.name
    pdf_ok = html_to_pdf(html, PDF_FILE)
    if pdf_ok:
        send_email(PDF_FILE, data['timestamp'])

print(f"\n{'═'*52}\n  Done — {data['timestamp']}\n{'═'*52}\n")


════════════════════════════════════════════════════
  Morning Market Dashboard — 2026-06-10 08:49 UTC
════════════════════════════════════════════════════

Indices...
FX...
Commodities...
Stocks...
US Treasury yields (Yahoo Finance)...
Bond yields (FRED)...
Euro Area yields (ECB SDW)...
Central bank rates (FRED)...

OK Data fetched.




(process:32405): GLib-CRITICAL **: 10:49:17.782: g_datalist_id_set_data_full: assertion 'key_id > 0' failed

(process:32405): GLib-GObject-CRITICAL **: 10:49:17.782: cannot unreference class of invalid (unclassed) type '(null)'

(process:32405): GLib-CRITICAL **: 10:49:17.783: g_datalist_id_set_data_full: assertion 'key_id > 0' failed

(process:32405): GLib-GObject-CRITICAL **: 10:49:17.783: cannot unreference class of invalid (unclassed) type '(null)'

(process:32405): GLib-CRITICAL **: 10:49:17.795: g_datalist_id_set_data_full: assertion 'key_id > 0' failed

(process:32405): GLib-GObject-CRITICAL **: 10:49:17.795: cannot unreference class of invalid (unclassed) type '(null)'

(process:32405): GLib-CRITICAL **: 10:49:17.795: g_datalist_id_set_data_full: assertion 'key_id > 0' failed

(process:32405): GLib-GObject-CRITICAL **: 10:49:17.795: cannot unreference class of invalid (unclassed) type '(null)'

(process:32405): GLib-CRITICAL **: 10:49:17.804: g_datalist_id_set_data_full: asser

✅ PDF generated: /var/folders/rz/sstbzcsx5cvfmclpx5zb548w0000gn/T/tmp_nwu_5cx.pdf
✅ Email sent to enzo.marichal@edu.escp.eu

════════════════════════════════════════════════════
  Done — 2026-06-10 08:49 UTC
════════════════════════════════════════════════════

